# Phase 1: MVTec AD 2 / Vial

OpenCV前処理 → PatchCore学習 → 推論 → Threshold → 評価 → 可視化 → 誤判定分析 → Run保存 を実行します。処理は `anomaly_detection` パッケージに実装されており、このNotebookは操作と結果確認に使います。

- 学習: `train/good`（正常のみ）
- Threshold決定: `test_public` を層化分割した **validation**
- 最終評価: 残りの **test**（Threshold決定には使わない）

## 1. Environment

In [ ]:
%matplotlib inline
from pathlib import Path

import pandas as pd
from IPython.display import display

from anomaly_detection.utils.config import find_project_root, load_config
from anomaly_detection.utils.device import describe_device, resolve_device
from anomaly_detection.utils.logging import setup_logging
from anomaly_detection.utils.reproducibility import collect_environment

setup_logging()
PROJECT_ROOT = find_project_root(Path.cwd())
env = collect_environment()
print("Python", env["python"], "|", env["platform"])
display(pd.Series(env["packages"], name="version").to_frame())

In [ ]:
# 利用可能なDevice（実行時は設定の runtime.device に従う）
describe_device(resolve_device("auto"))

## 2. Config

In [ ]:
CONFIG_PATH = PROJECT_ROOT / "configs" / "phase1_mvtec_vial.yaml"
config = load_config(CONFIG_PATH)
print("Config:", config.config_path)
config.to_dict()

## 3. Dataset Overview

Datasetが未配置の場合は配置方法を表示し、以降のセルはスキップします。

In [ ]:
from anomaly_detection.datasets import create_adapter
from anomaly_detection.datasets.base import DatasetNotFoundError, count_samples, read_image

adapter = create_adapter(config)
try:
    adapter.validate()
    DATA_READY = True
except DatasetNotFoundError as e:
    DATA_READY = False
    print(e)

if DATA_READY:
    samples = adapter.load_samples()
    display(pd.DataFrame(count_samples(samples)).T)

In [ ]:
from matplotlib.figure import Figure


def show_samples(rows: pd.DataFrame, title: str) -> Figure:
    figure = Figure(figsize=(3 * len(rows), 3.6))
    axes = figure.subplots(1, len(rows), squeeze=False)[0]
    for ax, (_, row) in zip(axes, rows.iterrows(), strict=True):
        ax.imshow(read_image(row["image_path"]))
        ax.set_title(Path(row["image_path"]).name, fontsize=8)
        ax.axis("off")
    figure.suptitle(title)
    return figure


if DATA_READY:
    train = samples[samples["split"] == "train"]
    defects = samples[samples["label"] == "defect"]
    display(show_samples(train.sample(4, random_state=0), "Sample: good (train)"))
    display(show_samples(defects.sample(4, random_state=0), "Sample: defect (test_public)"))

## 4. Preprocessing

前処理のON/OFFとパラメータは `configs/phase1_mvtec_vial.yaml` の `preprocessing` で変更します。処理順序は crop → resize → denoise → contrast → CLAHE → normalize で固定です。

In [ ]:
from anomaly_detection.preprocessing.pipeline import ImagePreprocessor, compare_images

preprocessor = ImagePreprocessor(config.preprocessing)
print("Preprocessing:", preprocessor.summary())
if DATA_READY:
    original = read_image(defects.iloc[0]["image_path"])
    display(compare_images(original, preprocessor.process(original)))

## 5. Training

`run_experiment` が次を1つのRunとして実行・保存します。

Device決定 → Seed設定 → Dataset読込 → PatchCore学習 → Model保存 → validation / testの推論 → Threshold決定（validation） → 評価（test） → Heatmap / Overlay保存 → `runs/summary.csv` へ追記

初回はBackbone（Wide ResNet50）の学習済み重みをDownloadします。

In [ ]:
from anomaly_detection.experiments.pipeline import run_experiment

result = run_experiment(config) if DATA_READY else None
if result:
    print("Run:", result.run.run_dir)

## 6. Inference

validation / test の推論結果です（`anomaly_score` はPatchCoreの生スコア）。

In [ ]:
if result:
    predictions = result.predictions
    display(predictions.drop(columns=["object_id"]).head(10))
    display(predictions.groupby(["split", "ground_truth"])["anomaly_score"].describe())

保存したModel Artifactを読み込み、任意のtest画像を推論できることを確認します。

In [ ]:
from anomaly_detection.inference.predictor import predict
from anomaly_detection.models.patchcore import load_patchcore

if result:
    model = load_patchcore(result.run.model_dir)
    targets = samples[samples["split"] == "test"].sample(4, random_state=1)
    device = resolve_device(config.runtime.device)
    reloaded = predict(model, targets, config, device, result.run.run_dir)
    display(reloaded.table[["image_path", "ground_truth", "anomaly_score"]])

## 7. Metrics

最終評価は **test** のみで算出します。validationの値はThreshold決定に使ったデータでの参考値です。外観検査では見逃し（False Negative）を重視し、Recallを主要指標として確認します。

In [ ]:
from anomaly_detection.evaluation.metrics import confusion_matrix_frame

if result:
    metrics = result.metrics
    print("Threshold:", metrics["threshold"])
    keys = [
        "recall",
        "precision",
        "f1",
        "accuracy",
        "auroc",
        "aupr",
        "num_images",
        "num_good",
        "num_defect",
    ]
    table = pd.DataFrame(
        {
            "test (final)": {k: metrics["image"][k] for k in keys},
            "validation (reference)": {
                k: metrics["threshold_selection_reference"]["image"][k] for k in keys
            },
        }
    )
    display(table)
    display(confusion_matrix_frame(metrics["image"]))
    print("Pixel-level (test):", metrics["pixel"])

## 8. Visualization

Original / Anomaly Heatmap / Overlay（Anomaly MapがThreshold以上の領域のContourとBounding Box）。全画像の結果はRunの `images/overlays/` と `images/heatmaps/` に保存されています。

In [ ]:
from anomaly_detection.visualization.anomaly_visualizer import load_display_image, render_result


def show_result(position: int) -> Figure:
    row = result.predictions.iloc[position]
    anomaly_map = result.anomaly_maps[position]
    image = load_display_image(row["image_path"], preprocessor, anomaly_map.shape)
    return render_result(image, anomaly_map, row, result.heatmap_scale)


if result:
    test_rows = result.predictions[result.predictions["split"] == "test"]
    for label in ("defect", "good"):
        position = test_rows[test_rows["ground_truth"] == label].index[0]
        display(show_result(position))

## 9. Error Analysis

testの誤判定を、見逃し（FN）→ 過検知（FP）の順に表示します。

In [ ]:
from anomaly_detection.evaluation.error_analysis import get_errors, plot_errors

if result:
    errors = get_errors(result.predictions, split="test", fn_first=True)
    display(errors["error_type"].value_counts())
    columns = ["error_type", "image_path", "ground_truth", "prediction", "anomaly_score"]
    display(errors[columns])

In [ ]:
if result:
    figures = plot_errors(
        errors, result.anomaly_maps, preprocessor, result.heatmap_scale, max_items=6
    )
    for figure in figures:
        display(figure)

### Thresholdを変更して再評価（再学習なし）

`NEW_THRESHOLD` に値（PatchCoreの生スコア単位）を設定して実行すると、保存済みスコアへ適用して新しいRunとして保存します（`retrained: false`）。

**注意:** testの結果を見ながら決めたThresholdのRunは、独立した最終評価ではありません。

In [ ]:
from anomaly_detection.experiments.pipeline import reevaluate

NEW_THRESHOLD = None  # 例: result.threshold.value * 0.9

if result and NEW_THRESHOLD is not None:
    re_result = reevaluate(result.run.run_dir, NEW_THRESHOLD)
    display(
        pd.DataFrame(
            {
                "before": result.metrics["image"],
                "after": re_result.metrics["image"],
            }
        ).loc[["recall", "precision", "f1", "accuracy", "fn", "fp"]]
    )

## 10. Experiment Result

In [ ]:
from anomaly_detection.experiments.run_manager import load_summary

if result:
    print("Run Directory:", result.run.run_dir)
    print(sorted(p.name for p in result.run.run_dir.iterdir()))
    display(load_summary(result.run.run_dir.parent).tail(5))